# Deploy Fabric Data Agent L400 workshop assets

This root deployment notebook downloads the workshop notebooks and semantic models from GitHub and publishes them to a Microsoft Fabric workspace.

## Items fetched from the repository

### Notebooks

- `NB_DataAgent_SDK_Setup_L400.ipynb`
- `NB_DataAgentEval_L400.ipynb`
- `NB_DataAgentEval_L400_AssistantAPI_TemporaryFix.ipynb`
- `NB_JudgeCalibration_L400.ipynb`
- `NB_OpsRefLakehouse_Build_and_Views_L400.ipynb`

### Semantic models

- `Manufacturing Ops.pbix`
- `Manufacturing Ops AI Ready.pbix`

- By default, assets are deployed to the workspace where this notebook is running.
- Set `WORKSPACE_ID` to deploy to another workspace.
- Existing notebooks are updated in place.
- Semantic models are published through the Power BI Imports REST API using `CreateOrOverwrite`, matching the standard PBIX publishing flow.

> The user running this notebook must have permission to create and update items in the target workspace.

## Step 1 - Deployment parameters

Leave `WORKSPACE_ID` blank to use the current workspace. This cell is marked as a parameter cell for pipeline execution.

In [ ]:
WORKSPACE_ID = ""
REPOSITORY_OWNER = "pawarbi"
REPOSITORY_NAME = "data-agent-L400-workshop"
REPOSITORY_REF = "main"

PUBLISH_NOTEBOOKS = True
PUBLISH_SEMANTIC_MODELS = True
OVERWRITE_EXISTING = True
REFRESH_SEMANTIC_MODELS = True

## Step 2 - Resolve the target workspace and asset manifest

In [ ]:
import io
import json
import time
from urllib.parse import quote
from uuid import UUID

import notebookutils
import requests
import sempy.fabric as fabric


NOTEBOOK_ASSETS = [
    {
        "path": "notebooks/NB_DataAgent_SDK_Setup_L400.ipynb",
        "description": "Creates and configures the multi-source Fabric data agent.",
    },
    {
        "path": "notebooks/NB_DataAgentEval_L400.ipynb",
        "description": "Runs the L400 Fabric data agent evaluation workflow.",
    },
    {
        "path": "notebooks/NB_DataAgentEval_L400_AssistantAPI_TemporaryFix.ipynb",
        "description": "Runs the temporary Assistant API evaluation workflow.",
    },
    {
        "path": "notebooks/NB_JudgeCalibration_L400.ipynb",
        "description": "Calibrates evaluation judges against human labels.",
    },
    {
        "path": "notebooks/NB_OpsRefLakehouse_Build_and_Views_L400.ipynb",
        "description": "Builds the OpsRefData Lakehouse and curated SQL views.",
    },
]

SEMANTIC_MODEL_ASSETS = [
    "semantic-models/Manufacturing Ops.pbix",
    "semantic-models/Manufacturing Ops AI Ready.pbix",
]


def validate_workspace_id(value):
    try:
        return str(UUID(str(value)))
    except ValueError as exc:
        raise ValueError(f"Invalid workspace ID: {value!r}") from exc


current_workspace_id = str(fabric.get_notebook_workspace_id())
target_workspace_id = validate_workspace_id(
    WORKSPACE_ID.strip() or current_workspace_id
)

print("Current workspace:", current_workspace_id)
print("Target workspace: ", target_workspace_id)
print("Repository:       ", f"{REPOSITORY_OWNER}/{REPOSITORY_NAME}@{REPOSITORY_REF}")

## Step 3 - Download and deployment helpers

In [ ]:
RAW_BASE_URL = (
    f"https://raw.githubusercontent.com/{REPOSITORY_OWNER}/"
    f"{REPOSITORY_NAME}/{REPOSITORY_REF}"
)


def download_asset(path):
    url = f"{RAW_BASE_URL}/{quote(path, safe='/')}"
    response = requests.get(
        url,
        headers={"User-Agent": "fabric-data-agent-l400-deployer"},
        timeout=180,
    )
    response.raise_for_status()
    content = response.content
    if content.startswith(b"version https://git-lfs.github.com/spec/v1"):
        raise RuntimeError(f"{path} resolved to a Git LFS pointer instead of file content.")
    return content


def deploy_notebook(asset, existing_notebooks):
    path = asset["path"]
    name = path.rsplit("/", 1)[-1].removesuffix(".ipynb")
    content = download_asset(path).decode("utf-8")
    parsed = json.loads(content)
    if parsed.get("nbformat") != 4 or not isinstance(parsed.get("cells"), list):
        raise ValueError(f"{path} is not a valid Jupyter notebook definition.")

    existing = existing_notebooks.get(name)
    if existing is None:
        created = notebookutils.notebook.create(
            name=name,
            description=asset["description"],
            content=content,
            workspaceId=target_workspace_id,
        )
        return {
            "type": "Notebook",
            "name": name,
            "action": "Created",
            "id": str(created.id),
        }

    if not OVERWRITE_EXISTING:
        raise FileExistsError(
            f"Notebook {name!r} already exists and OVERWRITE_EXISTING is False."
        )

    updated = notebookutils.notebook.updateDefinition(
        name=str(existing.id),
        content=content,
        workspaceId=target_workspace_id,
    )
    if not updated:
        raise RuntimeError(f"Fabric did not confirm the update of notebook {name!r}.")
    return {
        "type": "Notebook",
        "name": name,
        "action": "Updated",
        "id": str(existing.id),
    }


def publish_pbix(path, access_token):
    file_name = path.rsplit("/", 1)[-1]
    pbix_content = download_asset(path)
    conflict_mode = "CreateOrOverwrite" if OVERWRITE_EXISTING else "Abort"
    import_url = (
        "https://api.powerbi.com/v1.0/myorg/groups/"
        f"{target_workspace_id}/imports"
        f"?datasetDisplayName={quote(file_name)}"
        f"&nameConflict={conflict_mode}"
    )
    response = requests.post(
        import_url,
        headers={"Authorization": f"Bearer {access_token}"},
        files={
            "file": (
                file_name,
                io.BytesIO(pbix_content),
                "application/octet-stream",
            )
        },
        timeout=600,
    )
    if response.status_code not in (200, 201, 202):
        raise RuntimeError(
            f"PBIX upload failed for {file_name}: HTTP {response.status_code} "
            f"{response.text}"
        )

    import_id = response.json()["id"]
    status_url = (
        "https://api.powerbi.com/v1.0/myorg/groups/"
        f"{target_workspace_id}/imports/{import_id}"
    )
    for _ in range(120):
        status_response = requests.get(
            status_url,
            headers={"Authorization": f"Bearer {access_token}"},
            timeout=60,
        )
        status_response.raise_for_status()
        import_result = status_response.json()
        state = import_result.get("importState")
        if state == "Succeeded":
            datasets = import_result.get("datasets", [])
            reports = import_result.get("reports", [])
            return {
                "type": "Semantic model",
                "name": file_name.removesuffix(".pbix"),
                "action": "Published",
                "id": datasets[0]["id"] if datasets else "",
                "reportUrl": reports[0].get("webUrl", "") if reports else "",
            }
        if state == "Failed":
            raise RuntimeError(
                f"PBIX import failed for {file_name}: {json.dumps(import_result)}"
            )
        time.sleep(5)

    raise TimeoutError(f"PBIX import timed out for {file_name} after 10 minutes.")

## Step 4 - Deploy the selected assets

Notebook deployment uses `notebookutils.notebook`. PBIX deployment uses the Power BI Imports REST API and waits for each import to complete.

In [ ]:
deployment_results = []

if PUBLISH_NOTEBOOKS:
    existing_notebooks = {
        notebook.displayName: notebook
        for notebook in notebookutils.notebook.list(
            workspaceId=target_workspace_id,
            maxResults=1000,
        )
    }
    for asset in NOTEBOOK_ASSETS:
        result = deploy_notebook(asset, existing_notebooks)
        deployment_results.append(result)
        print(f"{result['action']}: {result['name']}")

if PUBLISH_SEMANTIC_MODELS:
    power_bi_token = notebookutils.credentials.getToken("pbi")
    for path in SEMANTIC_MODEL_ASSETS:
        result = publish_pbix(path, power_bi_token)
        deployment_results.append(result)
        print(f"{result['action']}: {result['name']}")

print("\nAsset publishing completed successfully.")

## Step 5 - Optionally refresh the semantic models

When `REFRESH_SEMANTIC_MODELS` is `True`, this step starts a full SemPy refresh for both models and waits for each refresh to complete. Set the parameter to `False` to skip this step.

In [ ]:
def refresh_semantic_model(model_name):
    print(f"Starting full refresh: {model_name}")
    request_id = fabric.refresh_dataset(
        dataset=model_name,
        workspace=target_workspace_id,
        refresh_type="full",
    )

    for _ in range(160):
        details = fabric.get_refresh_execution_details(
            dataset=model_name,
            refresh_request_id=request_id,
            workspace=target_workspace_id,
        )
        if details.status == "Completed":
            print(f"Refresh completed: {model_name}")
            return {
                "type": "Semantic model refresh",
                "name": model_name,
                "action": "Refreshed",
                "id": str(request_id),
            }
        if details.status in {"Failed", "Cancelled"}:
            messages = ""
            if details.messages is not None and not details.messages.empty:
                messages = "\n".join(
                    details.messages["Message"].astype(str).tolist()
                )
            raise RuntimeError(
                f"Refresh {details.status.lower()} for {model_name}. {messages}"
            )
        time.sleep(15)

    raise TimeoutError(
        f"Refresh timed out for {model_name} after 40 minutes. "
        f"Request ID: {request_id}"
    )


if REFRESH_SEMANTIC_MODELS:
    for path in SEMANTIC_MODEL_ASSETS:
        model_name = path.rsplit("/", 1)[-1].removesuffix(".pbix")
        deployment_results.append(refresh_semantic_model(model_name))
else:
    print("Semantic model refresh skipped because REFRESH_SEMANTIC_MODELS is False.")

## Step 6 - Deployment summary

In [ ]:
import pandas as pd

deployment_summary = pd.DataFrame(deployment_results)
display(deployment_summary)